# 01b — RavenPack **firm-level** news extraction, mapped to sectors**Academic research only — not investment advice.**`01_ravenpack_news_extraction.ipynb` pulls from `rpna.rpa_djpr_global_macro_<year>`. That table ismacro by construction: every event is about the economy, not about a company, so the resultingsentiment score is **one number per day shared by all eleven sector ETFs**. Measured on the currentpanel, the within-date standard deviation of every sentiment feature across the eleven tickers isexactly `0.000000`. The effective sample is ~1,500 days, not 16,588 panel rows, and no model canproduce a cross-sectional prediction from a feature with no cross-sectional variation.This notebook pulls the **equities** table instead, attributes each event to a company, maps thatcompany to one of the eleven sectors via SIC code, and aggregates to a`(session_date, sector)` panel — sentiment that actually differs across sectors on the same day.**Two design rules carried over from the diagnosis in notebook 11:**1. **Re-aligned window.** The 16:00 ET signal-date convention is kept identical to notebook 01 so the   two sources stay comparable, and the arrival timestamp is preserved so the modelling notebooks can   restrict to the non-leaky pre-open window `(close d−1, 09:30 d]`.2. **Ticker fixed effects are mandatory downstream.** Eleven ticker dummies alone, with zero news,   lift 5-day cross-sectional AUC by +0.0145 — larger than any sentiment effect measured in this   project so far. Any model built on this panel must put ticker FE in the *baseline* or it will   report that confound as a sentiment finding.**Licensing.** Row-level output goes only to `data_collection/raw/` (gitignored), same treatment asevery other WRDS extract in this repo.---> **This notebook has not been executed.** It was written without a WRDS session, so the table and> column names in Section 1 are inferred from the schema of the macro table and of> `rpna.rpa_source_list`. **Run Section 1 first** — it discovers what actually exists and prints it.> If a name differs, fix it in the constants cell and the rest of the notebook follows.

In [ ]:
%pip install -q wrds psycopg2-binary pandas numpy pyarrow

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import wrds

warnings.filterwarnings("ignore", category=FutureWarning)
pd.options.display.max_columns = 120

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "data_collection").is_dir():
    REPO_ROOT = PROJECT_ROOT
    NOTEBOOK_DIR = REPO_ROOT / "data_collection"
elif PROJECT_ROOT.name == "data_collection" and (PROJECT_ROOT.parent / "data_collection").is_dir():
    REPO_ROOT = PROJECT_ROOT.parent
    NOTEBOOK_DIR = PROJECT_ROOT
else:
    raise FileNotFoundError("Run this notebook from the repository root or data_collection/.")
RAW_DIR = NOTEBOOK_DIR / "raw"
RAW_DIR.mkdir(exist_ok=True)

START_DATE = "2020-01-01"
END_DATE = "2026-12-31"
YEARS = range(2020, 2027)

RELEVANCE_MIN = 90          # identical to notebook 01
EVENT_RELEVANCE_MIN = 90
MARKET_CLOSE_ET = "16:00:00"

# --- names to CONFIRM in Section 1 before running Section 2 ---
EQUITY_TABLE_TEMPLATE = "rpna.rpa_djpr_equities_{year}"
ENTITY_MAP_TABLE = "rpna.rpa_entity_mapping"

SECTOR_ETF_TO_ASSET = {
    "XLK": "Technology", "XLV": "Health_Care", "XLF": "Financials",
    "XLC": "Communication_Services", "XLY": "Consumer_Discretionary",
    "XLI": "Industrials", "XLP": "Consumer_Staples", "XLE": "Energy",
    "XLU": "Utilities", "XLB": "Materials", "XLRE": "Real_Estate",
}

EQUITY_EVENTS_CSV = RAW_DIR / f"ravenpack_equity_events_{START_DATE[:4]}_{END_DATE[:4]}.csv"
SECTOR_NEWS_CSV = NOTEBOOK_DIR / "sector_news_daily_df.csv"

print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Silver output (gitignored): {EQUITY_EVENTS_CSV}")
print(f"Gold output (committed):    {SECTOR_NEWS_CSV}")

## 1. Schema discovery — **run this first**Confirms the equities tables and the entity-mapping table exist and prints their columns. Nothingbelow this section will work until the names printed here match the constants above.

In [ ]:
db = wrds.Connection()

rpna_tables = db.list_tables(library="rpna")
print(f"rpna library: {len(rpna_tables)} tables\n")

equity_like = sorted(t for t in rpna_tables if "equit" in t.lower())
macro_like = sorted(t for t in rpna_tables if "macro" in t.lower())
map_like = sorted(t for t in rpna_tables if "map" in t.lower() or "entity" in t.lower())

print("Equity-style tables :", equity_like[:12] or "NONE FOUND")
print("Macro tables (ref)  :", macro_like[:12])
print("Entity/mapping      :", map_like[:12] or "NONE FOUND")

In [ ]:
probe_table = EQUITY_TABLE_TEMPLATE.format(year=2020).split(".", 1)[1]
if probe_table not in rpna_tables:
    raise RuntimeError(
        f"'{probe_table}' not in rpna. Pick the right name from the 'Equity-style tables' list "
        f"printed above and update EQUITY_TABLE_TEMPLATE."
    )

equity_cols = db.describe_table(library="rpna", table=probe_table)
print(f"--- columns of rpna.{probe_table} ---")
display(equity_cols)

needed = ["rp_story_id", "timestamp_utc", "relevance", "event_relevance", "rp_source_id",
          "topic", "group", "event_sentiment_score", "rp_entity_id"]
present = set(equity_cols["name"].str.lower())
missing = [c for c in needed if c.lower() not in present]
print("\nMissing expected columns:", missing or "none — the query in Section 2 should run as written")

In [ ]:
map_probe = ENTITY_MAP_TABLE.split(".", 1)[1]
if map_probe in rpna_tables:
    display(db.describe_table(library="rpna", table=map_probe))
    print(f"\nSample rows of rpna.{map_probe}:")
    display(db.raw_sql(f"SELECT * FROM rpna.{map_probe} LIMIT 8"))
else:
    print(f"'{map_probe}' not found. Candidates containing 'map' or 'entity':")
    for t in map_like:
        print("   ", t)
    print("\nPick the one holding rp_entity_id -> CUSIP/ISIN/TICKER and set ENTITY_MAP_TABLE.")

## 2. Pull firm-level eventsSame filters as notebook 01 — relevance ≥ 90, event relevance ≥ 90, rank-1 non-blog institutionalsources only, identical 16:00 ET signal-date rule — so the macro and equity extracts differ *only*in the underlying table. That is what makes the eventual macro-vs-firm comparison clean.

In [ ]:
def sql_string_list(values):
    """Return a SQL-safe single-quoted literal list for simple identifier strings."""
    return ", ".join("'" + str(v).replace("'", "''") + "'" for v in values)


source_attrs = db.raw_sql("""
    SELECT rp_entity_id, data_type, data_value
    FROM rpna.rpa_source_list
    WHERE data_type IN ('ENTITY_NAME', 'PUBLICATION_TYPE', 'SOURCE_RANK')
""")
sources = (source_attrs.pivot(index="rp_entity_id", columns="data_type", values="data_value")
                       .reset_index()
                       .rename(columns={"ENTITY_NAME": "source_name",
                                        "PUBLICATION_TYPE": "source_type",
                                        "SOURCE_RANK": "source_rank"}))
sources.columns.name = None
sources["source_rank"] = pd.to_numeric(sources["source_rank"], errors="coerce")
institutional = sources.loc[sources["source_rank"].eq(1)
                            & sources["source_type"].notna()
                            & sources["source_type"].ne("BLOG")]
source_id_sql = sql_string_list(institutional["rp_entity_id"].dropna().astype(str))
print(f"Rank-1 non-blog institutional sources: {len(institutional):,}")

In [ ]:
ET_TS = "((timestamp_utc AT TIME ZONE 'UTC') AT TIME ZONE 'America/New_York')"


def fetch_equity_events(year):
    table = EQUITY_TABLE_TEMPLATE.format(year=year)
    query = f"""
        SELECT
            rp_story_id,
            rp_entity_id,
            timestamp_utc,
            CASE
                WHEN {ET_TS}::time < TIME '{MARKET_CLOSE_ET}'
                    THEN {ET_TS}::date
                ELSE ({ET_TS}::date + INTERVAL '1 day')::date
            END AS signal_calendar_date,
            relevance,
            event_relevance,
            rp_source_id,
            topic,
            "group" AS group_name,
            event_sentiment_score
        FROM {table}
        WHERE rpa_date_utc BETWEEN DATE '{START_DATE}' AND DATE '{END_DATE}'
          AND relevance >= {RELEVANCE_MIN}
          AND event_relevance >= {EVENT_RELEVANCE_MIN}
          AND rp_source_id IN ({source_id_sql})
          AND timestamp_utc IS NOT NULL
          AND event_sentiment_score IS NOT NULL
          AND rp_entity_id IS NOT NULL
    """
    df = db.raw_sql(query)
    print(f"  {year}: {len(df):,} events")
    return df


# Headline/event_text are deliberately NOT selected here: sector attribution needs only the
# structured fields, and leaving the licensed text out keeps this extract lighter to handle.
equity_events = pd.concat([fetch_equity_events(y) for y in YEARS], ignore_index=True)
print(f"\nTotal firm-level events: {len(equity_events):,}")
print(f"Distinct entities: {equity_events['rp_entity_id'].nunique():,}")

## 3. Attribute each entity to a sector`rp_entity_id → CUSIP → CRSP permno → SIC code → sector`. CUSIP is preferred over ticker becausetickers are reused across time and across companies; the CRSP security-history table is date-aware,so the link is point-in-time rather than as-of-today.

In [ ]:
entity_ids = equity_events["rp_entity_id"].dropna().unique().tolist()
print(f"Resolving {len(entity_ids):,} entities...")

ent_map = db.raw_sql(f"""
    SELECT rp_entity_id, data_type, data_value
    FROM {ENTITY_MAP_TABLE}
    WHERE data_type IN ('CUSIP', 'ISIN', 'TICKER', 'ENTITY_NAME')
""")
ent_wide = (ent_map.pivot_table(index="rp_entity_id", columns="data_type",
                                values="data_value", aggfunc="first")
                   .reset_index())
ent_wide.columns.name = None
print("Entity attributes available:", [c for c in ent_wide.columns if c != "rp_entity_id"])

# CUSIP-8 is the CRSP join key (RavenPack usually carries CUSIP-9; ISIN embeds CUSIP at chars 3-11).
if "CUSIP" in ent_wide:
    ent_wide["cusip8"] = ent_wide["CUSIP"].astype(str).str.strip().str[:8].str.upper()
elif "ISIN" in ent_wide:
    ent_wide["cusip8"] = ent_wide["ISIN"].astype(str).str.strip().str[2:10].str.upper()
else:
    ent_wide["cusip8"] = np.nan
    print("WARNING: no CUSIP or ISIN available — the notebook will fall back to ticker matching.")

print(f"Entities with a usable CUSIP-8: {ent_wide['cusip8'].notna().sum():,}")

In [ ]:
crsp_sec = db.raw_sql(f"""
    SELECT DISTINCT permno, ticker, cusip9, siccd, secinfostartdt, secinfoenddt
    FROM crsp.stksecurityinfohist
    WHERE secinfostartdt <= DATE '{END_DATE}'
      AND COALESCE(secinfoenddt, DATE '9999-12-31') >= DATE '{START_DATE}'
""")
crsp_sec["cusip8"] = crsp_sec["cusip9"].astype(str).str.strip().str[:8].str.upper()
crsp_sec["siccd"] = pd.to_numeric(crsp_sec["siccd"], errors="coerce")
crsp_link = (crsp_sec.dropna(subset=["cusip8", "siccd"])
                     .sort_values("secinfostartdt")
                     .drop_duplicates("cusip8", keep="last")[["cusip8", "permno", "ticker", "siccd"]])

linked = ent_wide.merge(crsp_link, on="cusip8", how="left")
if linked["siccd"].isna().mean() > 0.5 and "TICKER" in ent_wide:
    tick_link = (crsp_sec.dropna(subset=["ticker", "siccd"])
                         .sort_values("secinfostartdt")
                         .drop_duplicates("ticker", keep="last")[["ticker", "permno", "siccd"]]
                         .rename(columns={"ticker": "TICKER", "permno": "permno_t", "siccd": "siccd_t"}))
    linked = linked.merge(tick_link, on="TICKER", how="left")
    linked["siccd"] = linked["siccd"].fillna(linked["siccd_t"])
    linked["permno"] = linked["permno"].fillna(linked.get("permno_t"))

print(f"Entities linked to a SIC code: {linked['siccd'].notna().sum():,} "
      f"({linked['siccd'].notna().mean():.1%})")

In [ ]:
# SIC -> the eleven SPDR sectors. Ranges are evaluated in order, so narrow rules precede wide ones.
SIC_TO_SECTOR = [
    ((2830, 2836), "Health_Care"),        ((3841, 3851), "Health_Care"),
    ((8000, 8099), "Health_Care"),
    ((3570, 3579), "Technology"),         ((3660, 3679), "Technology"),
    ((7370, 7379), "Technology"),         ((3812, 3827), "Technology"),
    ((2700, 2799), "Communication_Services"), ((4800, 4899), "Communication_Services"),
    ((7800, 7999), "Communication_Services"),
    ((6500, 6599), "Real_Estate"),        ((6798, 6798), "Real_Estate"),
    ((4900, 4999), "Utilities"),
    ((1200, 1399), "Energy"),             ((2900, 2999), "Energy"),
    ((6000, 6499), "Financials"),         ((6700, 6799), "Financials"),
    ((2000, 2199), "Consumer_Staples"),   ((5400, 5499), "Consumer_Staples"),
    ((5912, 5912), "Consumer_Staples"),
    ((1000, 1099), "Materials"),          ((1400, 1499), "Materials"),
    ((2600, 2699), "Materials"),          ((2800, 2829), "Materials"),
    ((2840, 2899), "Materials"),          ((3000, 3099), "Materials"),
    ((3200, 3399), "Materials"),
    ((3711, 3716), "Consumer_Discretionary"), ((2200, 2399), "Consumer_Discretionary"),
    ((5200, 5999), "Consumer_Discretionary"), ((7000, 7299), "Consumer_Discretionary"),
    ((3900, 3999), "Consumer_Discretionary"),
    ((1500, 1799), "Industrials"),        ((3400, 3569), "Industrials"),
    ((3580, 3659), "Industrials"),        ((3700, 3799), "Industrials"),
    ((4000, 4799), "Industrials"),        ((8700, 8799), "Industrials"),
]


def sic_to_sector(sic):
    if pd.isna(sic):
        return np.nan
    sic = int(sic)
    for (lo, hi), sector in SIC_TO_SECTOR:
        if lo <= sic <= hi:
            return sector
    return np.nan


linked["sector"] = linked["siccd"].map(sic_to_sector)
entity_sector = linked.dropna(subset=["sector"])[["rp_entity_id", "sector"]]
print(f"Entities mapped to a sector: {len(entity_sector):,}")
print(entity_sector["sector"].value_counts().to_string())

## 4. Sector-day panel — and the check that decides whether this was worth doing

In [ ]:
attributed = equity_events.merge(entity_sector, on="rp_entity_id", how="inner")
print(f"Events attributed to a sector: {len(attributed):,} "
      f"({len(attributed)/len(equity_events):.1%} of firm-level events)")

attributed.to_csv(EQUITY_EVENTS_CSV, index=False)
print(f"Saved silver -> {EQUITY_EVENTS_CSV}  (gitignored)")

attributed["signal_calendar_date"] = pd.to_datetime(attributed["signal_calendar_date"])
grp = attributed.groupby(["signal_calendar_date", "sector"])
sector_news = pd.DataFrame({
    "event_count": grp.size(),
    "mean_event_sentiment_score": grp["event_sentiment_score"].mean(),
    "net_event_sentiment": grp["event_sentiment_score"].sum(),
    "sentiment_dispersion": grp["event_sentiment_score"].std(),
    "positive_event_share": grp["event_sentiment_score"].apply(lambda s: (s > 0.3).mean()),
    "negative_event_share": grp["event_sentiment_score"].apply(lambda s: (s < -0.3).mean()),
}).reset_index().rename(columns={"signal_calendar_date": "session_date"})

sector_news.to_csv(SECTOR_NEWS_CSV, index=False)
print(f"Saved gold  -> {SECTOR_NEWS_CSV}  [{len(sector_news):,} sector-day rows]")

In [ ]:
# THE decisive check. The macro panel scores 0.000000 here. Anything above ~0.02 means this
# extract has done its job and a cross-sectional model is finally possible.
within_date_std = sector_news.groupby("session_date")["mean_event_sentiment_score"].std().mean()
overall_std = sector_news["mean_event_sentiment_score"].std()

print(f"mean within-date std across sectors : {within_date_std:.6f}   <- macro panel = 0.000000")
print(f"overall std                         : {overall_std:.6f}")
print(f"share of variation that is cross-sectional : "
      f"{within_date_std**2 / overall_std**2:.1%}")

coverage = sector_news.groupby("sector").agg(
    days=("session_date", "nunique"),
    median_events_per_day=("event_count", "median"),
    mean_sentiment=("mean_event_sentiment_score", "mean"),
).sort_values("days")
display(coverage)

thin = coverage[coverage["median_events_per_day"] < 3]
if len(thin):
    print("\nWARNING - thin coverage, these sectors will be noisy:")
    print(thin.to_string())

## Handoff- `data_collection/raw/ravenpack_equity_events_2020_2026.csv` — silver, gitignored, one row per  firm-level event with its sector, arrival timestamp and signal date.- `data_collection/sector_news_daily_df.csv` — gold, one row per `(session_date, sector)`.**Before modelling on this panel, two non-negotiables:**1. **Ticker fixed effects belong in the baseline.** Eleven ticker dummies with no news at all lift   5-day cross-sectional AUC by +0.0145. A model without ticker FE in its baseline will report that   as a sentiment result. This was confirmed empirically earlier in the project: the first   permutation null centred at +0.0047 instead of 0 for exactly this reason.2. **Restrict to the pre-open window.** `timestamp_utc` is preserved above precisely so the modelling   notebook can keep only events in `(close d−1, 09:30 d]`. Using the full signal-date window   reproduces the already-priced feature that produced the null in notebooks 05–10.**Then** re-run the notebook 11 grid with sector as the cross-sectional unit, and compare the resultto the macro null on the same axis.